# 📊 Demo AFP Capital — Tipo de Cambio desde Banco Central de Chile
**Sesión 5: Analítica de Datos con Power BI y SQL**  
**Instructor:** Walter Calcagno · Microsoft MVP Data Platform  

---
### ¿Qué vamos a hacer?
1. 🕸️ **Scraping** de la página pública del Banco Central de Chile  
2. 🔄 **Unpivot** (melt): tabla Año × Mes → formato largo  
3. 📈 **Visualización**: serie histórica + heatmap  
4. 🔮 **Predicción**: Media Móvil para los próximos 3 meses  

> **Fuente:** https://si3.bcentral.cl — Indicadores Diarios — Tipo de Cambio Observado (CLP/USD)


## 1️⃣ Instalación

In [ ]:
%%capture
# Librerías principales
!pip install requests beautifulsoup4 lxml pandas matplotlib seaborn plotly --quiet

# Selenium + ChromeDriver (fallback si la página usa JavaScript)
!apt-get install -y chromium-chromedriver --quiet 2>/dev/null
!pip install selenium --quiet

print("✅ Todo instalado")

## 2️⃣ Importaciones

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
from bs4 import BeautifulSoup
from io import StringIO
from datetime import datetime, timedelta
import warnings, re, time

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 12})

print("✅ Imports OK —", datetime.today().strftime('%Y-%m-%d %H:%M'))

## 3️⃣ URL pública del Banco Central de Chile

Esta URL apunta directamente a la serie **Tipo de Cambio Observado (USD/CLP)**  
en la sección de Indicadores Diarios del Banco Central.


In [ ]:
URL = (
    "https://si3.bcentral.cl/Indicadoressiete/secure/Serie.aspx"
    "?gcode=PRE_TCO"
    "&param=RABmAFYAWQB3AGYAaQBuAEkALQAzADUAbgBNAGgAaAAkADUAVwBQAC4AbQBY"
    "ADAARwBOAGUAYwBjACMAQQBaAHAARgBhAGcAUABTAGUAdwA1ADQAMQA0AE0AawBLAF8A"
    "dQBDACQASABzAG0AXwA2AHQAawBvAFcAZwBKAEwAegBzAF8AbgBMAHIAYgBDAC4ARQA3"
    "AFUAVwB4AFIAWQBhAEEAOABkAHkAZwAxAEEARAA="
)

print("🌐 URL objetivo:")
print(f"   {URL[:80]}...")

## 4️⃣ Scraping de la página

Intentamos primero con `requests` (rápido). Si la página requiere JavaScript  
activamos Selenium con Chrome headless automáticamente.


In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    "Referer": "https://si3.bcentral.cl/",
}

def scrape_with_requests(url):
    """Intento 1: requests + BeautifulSoup (sin JS)."""
    print("🔄 Intentando con requests...")
    session = requests.Session()
    resp = session.get(url, headers=HEADERS, timeout=30, verify=False)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")
    tables = pd.read_html(StringIO(str(soup)), flavor="lxml")
    return tables, resp.text

def scrape_with_selenium(url):
    """Intento 2: Selenium headless Chrome (con JS)."""
    print("🔄 Activando Selenium (Chrome headless)...")
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument(f"user-agent={HEADERS['User-Agent']}")

    service = Service("/usr/bin/chromedriver")
    driver  = webdriver.Chrome(service=service, options=opts)

    try:
        driver.get(url)
        # Esperar hasta que haya al menos una tabla con datos
        WebDriverWait(driver, 45).until(
            EC.presence_of_element_located((By.TAG_NAME, "table"))
        )
        time.sleep(3)  # render extra
        html = driver.page_source
    finally:
        driver.quit()

    tables = pd.read_html(StringIO(html), flavor="lxml")
    return tables, html

# ── Ejecutar scraping ─────────────────────────────────────────────────────────
html_raw = None
all_tables = []

try:
    all_tables, html_raw = scrape_with_requests(URL)
    # Validar que hay tablas útiles (con números, más de 5 filas)
    useful = [t for t in all_tables if t.shape[0] > 5 and t.select_dtypes(include='number').shape[1] > 0]
    if not useful:
        raise ValueError("requests OK pero sin tablas numéricas útiles → usando Selenium")
    print(f"✅ requests exitoso — {len(all_tables)} tabla(s) encontrada(s)")

except Exception as e:
    print(f"⚠️  {e}")
    all_tables, html_raw = scrape_with_selenium(URL)
    print(f"✅ Selenium exitoso — {len(all_tables)} tabla(s) encontrada(s)")

# Mostrar resumen de tablas disponibles
print()
print("📋 Tablas encontradas:")
for i, t in enumerate(all_tables):
    print(f"   [{i}] shape={t.shape}  columnas={list(t.columns[:6])}")

## 5️⃣ Seleccionar y limpiar la tabla de datos

El Banco Central muestra los datos en formato **PIVOT** (Año × Mes).  
Vamos a identificar esa tabla y limpiarla.


In [ ]:
def seleccionar_tabla_bcch(tables):
    """
    Selecciona la tabla principal de datos:
    busca la tabla más grande con columnas de meses (Ene/Feb/Enero/January...).
    """
    meses_patron = re.compile(
        r'(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic'
        r'|enero|febrero|marzo|abril|mayo|junio|julio|agosto'
        r'|septiembre|octubre|noviembre|diciembre'
        r'|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)',
        re.IGNORECASE
    )

    mejor = None
    mejor_score = 0

    for t in tables:
        cols_str = ' '.join(str(c) for c in t.columns).lower()
        score = len(meses_patron.findall(cols_str))
        if score > mejor_score and t.shape[0] > 2:
            mejor_score = score
            mejor = t

    # Si no encontramos por meses, tomar la tabla numérica más grande
    if mejor is None:
        numericas = [t for t in tables if t.select_dtypes(include='number').shape[1] >= 3]
        if numericas:
            mejor = max(numericas, key=lambda t: t.shape[0] * t.shape[1])

    return mejor

df_pivot_raw = seleccionar_tabla_bcch(all_tables)

if df_pivot_raw is None:
    raise ValueError("❌ No se encontró tabla de datos. Revisa all_tables manualmente.")

print("✅ Tabla seleccionada:")
print(f"   Shape: {df_pivot_raw.shape}")
print()
df_pivot_raw

## 6️⃣ Limpieza del pivot

In [ ]:
# ─── LIMPIEZA DE LA TABLA SCRAPEADA ──────────────────────────────────────────
# La tabla viene en formato CALENDARIO: filas = días del mes, columnas = meses
# Columna 0 = "Día",  columnas 1-12 = Enero ... Diciembre
# Los valores están en centésimas de peso  →  ÷ 100 para obtener CLP/USD real

YEAR_DATOS = datetime.today().year   # año en curso (2026)

day_col  = df_pivot_raw.columns[0]                 # "Día"
mes_cols = list(df_pivot_raw.columns[1:])          # ['Enero', ..., 'Diciembre']

df_raw = df_pivot_raw.copy().rename(columns={day_col: 'Dia'})

# Conservar solo filas con día numérico (elimina eventuales filas de encabezado)
df_raw['Dia'] = pd.to_numeric(df_raw['Dia'], errors='coerce')
df_raw = df_raw.dropna(subset=['Dia'])
df_raw['Dia'] = df_raw['Dia'].astype(int)

# Convertir valores: texto → float y dividir entre 100
for col in mes_cols:
    df_raw[col] = (df_raw[col].astype(str)
                   .str.replace(',', '.', regex=False)
                   .str.replace(r'[^\d.]', '', regex=True))
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce') / 100.0

print(f"✅ Tabla limpia: {df_raw.shape[0]} días × {len(mes_cols)} meses")
print(f"   Año: {YEAR_DATOS}")
print(f"   Rango de valores: {df_raw[mes_cols].stack().min():.2f}–{df_raw[mes_cols].stack().max():.2f} CLP/USD")
print()
df_raw.head(8)


## 7️⃣ UNPIVOT — de Ancho a Largo con `melt()`

El **unpivot** convierte las **columnas de meses** en filas:  
- **Antes (Wide):** `Año | Ene | Feb | Mar | ... | Dic`  
- **Después (Long):** `Año | Mes | CLP_USD`  

Este es el formato ideal para graficar, analizar y modelar.


In [ ]:
# ─── UNPIVOT: Ancho → Largo con melt() ───────────────────────────────────────
# ANTES  (Wide):  Dia | Enero | Febrero | ... | Diciembre
# DESPUÉS (Long): Fecha | Mes  | CLP_USD

MESES_NUM = {
    'Enero':1,'Febrero':2,'Marzo':3,'Abril':4,'Mayo':5,'Junio':6,
    'Julio':7,'Agosto':8,'Septiembre':9,'Octubre':10,'Noviembre':11,'Diciembre':12
}

# melt(): convierte cada columna-mes en una fila
df_long = (
    df_raw
    .melt(id_vars='Dia', var_name='Mes', value_name='CLP_USD')
    .dropna(subset=['CLP_USD'])
    .copy()
)

# Número de mes y fecha real
df_long['Mes_Num'] = df_long['Mes'].map(MESES_NUM)
df_long['Fecha'] = pd.to_datetime(
    dict(year=YEAR_DATOS, month=df_long['Mes_Num'], day=df_long['Dia']),
    errors='coerce'
)

# Descartar fechas inválidas (ej: 31 de febrero) y días futuros
df_long = (df_long
           .dropna(subset=['Fecha'])
           .query('Fecha <= @pd.Timestamp.today()')
           .sort_values('Fecha')
           .reset_index(drop=True)
           [['Fecha','Mes','Mes_Num','CLP_USD']])

print(f"✅ Unpivot completado: {len(df_long):,} registros diarios")
print(f"   Período : {df_long['Fecha'].min():%d-%b-%Y}  →  {df_long['Fecha'].max():%d-%b-%Y}")
print(f"   Promedio: {df_long['CLP_USD'].mean():.2f} CLP/USD")
print()
print("Formato LARGO (primeras filas):")
df_long.head(8)


## 8️⃣ Visualización

In [ ]:
# ─── VISUALIZACIÓN — Serie diaria + Promedio mensual ─────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# ── Gráfico 1: Serie diaria ───────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(df_long['Fecha'], df_long['CLP_USD'],
         color='#1565C0', linewidth=1.3, alpha=0.85, label='CLP/USD diario')
ax1.fill_between(df_long['Fecha'], df_long['CLP_USD'],
                 df_long['CLP_USD'].min() * 0.99,
                 alpha=0.1, color='#1565C0')

idx_max = df_long['CLP_USD'].idxmax()
idx_min = df_long['CLP_USD'].idxmin()
for idx, label, color, dy in [(idx_max,'Máximo','#C62828',20),
                               (idx_min,'Mínimo','#2E7D32',-35)]:
    ax1.annotate(
        f"{label}\n${df_long.loc[idx,'CLP_USD']:.2f}",
        xy=(df_long.loc[idx,'Fecha'], df_long.loc[idx,'CLP_USD']),
        xytext=(15, dy), textcoords='offset points',
        fontsize=9, color=color, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=color, lw=1.5)
    )

ax1.set_title(f'Tipo de Cambio Observado CLP/USD — Datos Diarios {YEAR_DATOS}',
              fontsize=13, fontweight='bold')
ax1.set_ylabel('CLP por 1 USD', fontsize=11)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.2f}'))
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# ── Gráfico 2: Promedio mensual ───────────────────────────────────────────────
ax2 = axes[1]
df_monthly = (df_long.groupby('Mes_Num')['CLP_USD']
              .mean().reset_index())
df_monthly['Mes_Nombre'] = df_monthly['Mes_Num'].map(
    {v:k for k,v in MESES_NUM.items()})

bars = ax2.bar(df_monthly['Mes_Nombre'], df_monthly['CLP_USD'],
               color='#1565C0', alpha=0.82, edgecolor='white', width=0.65)
for bar, val in zip(bars, df_monthly['CLP_USD']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'${val:.1f}', ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax2.set_title(f'Promedio Mensual CLP/USD — {YEAR_DATOS}', fontsize=12, fontweight='bold')
ax2.set_ylabel('CLP por 1 USD', fontsize=11)
ax2.set_xlabel('Mes', fontsize=11)
rng = df_monthly['CLP_USD'].max() - df_monthly['CLP_USD'].min()
ax2.set_ylim(df_monthly['CLP_USD'].min() - rng, df_monthly['CLP_USD'].max() + rng)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.1f}'))
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('viz_serie_diaria.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: viz_serie_diaria.png")


### 🗓️ Heatmap Calendario (formato pivot original)

In [ ]:
# ─── HEATMAP CALENDARIO — Día × Mes (datos originales antes del unpivot) ──────
# Muestra el formato PIVOT original: filas = día del mes, columnas = mes

# Solo meses con al menos un valor
meses_con_datos = [m for m in mes_cols if df_raw.set_index('Dia')[m].notna().any()]
pivot_heatmap = df_raw.set_index('Dia')[meses_con_datos].copy()

fig, ax = plt.subplots(figsize=(len(meses_con_datos) * 1.6 + 2, 11))

sns.heatmap(
    pivot_heatmap.astype(float),
    annot=True, fmt='.1f',
    cmap='RdYlGn_r',
    linewidths=0.35, linecolor='#e8e8e8',
    cbar_kws={'label': 'CLP / USD', 'shrink': 0.55},
    ax=ax, annot_kws={'size': 7.5},
    mask=pivot_heatmap.isnull()
)
ax.set_title(
    f'Heatmap CLP/USD — Tipo de Cambio Diario {YEAR_DATOS}\n'
    f'(Este es el formato PIVOT que vamos a transformar con melt)',
    fontsize=13, fontweight='bold', pad=15
)
ax.set_xlabel('Mes', fontsize=11)
ax.set_ylabel('Día del mes', fontsize=11)
plt.tight_layout()
plt.savefig('heatmap_calendario.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Heatmap guardado: heatmap_calendario.png")


## 9️⃣ Predicción — Media Móvil para los próximos 3 meses

### Metodología
| Paso | Detalle |
|------|---------|
| 1 | Calcular Media Móvil de 6 meses sobre datos históricos |
| 2 | Estimar tendencia lineal (slope) de los últimos 12 meses |
| 3 | Proyectar 3 meses hacia adelante |
| 4 | Banda de confianza: ±1.5 desviaciones estándar |


In [ ]:
# ─── PREDICCIÓN — Media Móvil + Proyección 3 meses ───────────────────────────
WINDOW_MA = 20      # días hábiles ≈ 1 mes de cotizaciones
HORIZON   = 66      # días a proyectar (≈ 3 meses calendario)
TREND_WIN = min(60, len(df_long) - WINDOW_MA)  # ventana para estimar pendiente

df_long = df_long.sort_values('Fecha').reset_index(drop=True)
df_long['MA20']  = df_long['CLP_USD'].rolling(window=WINDOW_MA, min_periods=5).mean()
df_long['STD20'] = df_long['CLP_USD'].rolling(window=WINDOW_MA, min_periods=5).std()

# Tendencia lineal sobre los últimos TREND_WIN puntos de la MA
ma_series = df_long['MA20'].dropna()
recent    = ma_series.tail(max(TREND_WIN, 10)).values
slope, _  = np.polyfit(np.arange(len(recent)), recent, 1)

ultimo_val   = df_long['CLP_USD'].iloc[-1]
ultimo_ma    = df_long['MA20'].dropna().iloc[-1]
ultima_std   = df_long['STD20'].dropna().iloc[-1]
ultima_fecha = df_long['Fecha'].iloc[-1]

# Fechas futuras (días hábiles, lunes-viernes)
future_all   = pd.date_range(ultima_fecha + pd.Timedelta(days=1), periods=HORIZON*2, freq='D')
future_dates = future_all[future_all.dayofweek < 5][:HORIZON]

proj_vals  = [ultimo_ma  + slope * (i+1) for i in range(len(future_dates))]
proj_upper = [v + 1.5 * ultima_std        for v in proj_vals]
proj_lower = [v - 1.5 * ultima_std        for v in proj_vals]

df_proj = pd.DataFrame({
    'Fecha':      future_dates,
    'Proyeccion': proj_vals,
    'Upper':      proj_upper,
    'Lower':      proj_lower,
})

print("✅ Predicción calculada:")
print(f"   Último valor real : ${ultimo_val:.2f}  ({ultima_fecha:%d-%b-%Y})")
print(f"   MA-{WINDOW_MA} actual      : ${ultimo_ma:.2f}")
print(f"   Tendencia diaria  : {slope:+.3f} CLP/día  ({slope*20:+.1f} CLP/mes)")
print()
print("   Proyección 3 meses:")
print(f"     Fecha fin       : {future_dates[-1]:%d-%b-%Y}")
print(f"     Estimado        : ${proj_vals[-1]:.2f}")
print(f"     Intervalo ±1.5σ : ${proj_lower[-1]:.2f}  —  ${proj_upper[-1]:.2f}")


### 📈 Visualización de la predicción

In [ ]:
# ─── GRÁFICO DE PREDICCIÓN — Matplotlib ──────────────────────────────────────
n_hist  = min(len(df_long), 80)
df_hist = df_long.tail(n_hist).copy()

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df_hist['Fecha'], df_hist['CLP_USD'],
        color='#90CAF9', linewidth=1.1, alpha=0.85,
        marker='o', markersize=2.5, label='CLP/USD diario')
ax.plot(df_hist['Fecha'], df_hist['MA20'],
        color='#1565C0', linewidth=2.8, label=f'MA-{WINDOW_MA} días (histórica)')
ax.fill_between(df_hist['Fecha'],
                df_hist['MA20'] - df_hist['STD20'],
                df_hist['MA20'] + df_hist['STD20'],
                alpha=0.1, color='#1565C0', label='Banda ±1σ histórica')

ax.plot(df_proj['Fecha'], df_proj['Proyeccion'],
        color='#FF6F00', linewidth=2.5, linestyle='--',
        marker='D', markersize=3.5, label='Proyección MA-20')
ax.fill_between(df_proj['Fecha'], df_proj['Lower'], df_proj['Upper'],
                alpha=0.15, color='#FF6F00', label='Intervalo ±1.5σ')

y_lim = ax.get_ylim()
ax.axvline(ultima_fecha, color='#9E9E9E', linestyle=':', linewidth=1.5)
ax.text(ultima_fecha, y_lim[1], '  ← Histórico | Proyección →',
        fontsize=9, color='#616161', va='top')

ax.set_title(f'CLP/USD — MA-{WINDOW_MA} días + Proyección 3 meses',
             fontsize=13, fontweight='bold')
ax.set_ylabel('CLP por 1 USD', fontsize=11)
ax.set_xlabel('Fecha', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.2f}'))
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('prediccion_clp_usd.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: prediccion_clp_usd.png")


### 🖱️ Versión interactiva con Plotly

In [ ]:
# ─── VERSIÓN INTERACTIVA — Plotly ────────────────────────────────────────────
import plotly.graph_objects as go

n_hist  = min(len(df_long), 80)
df_hist = df_long.tail(n_hist).copy()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_hist['Fecha'], y=df_hist['CLP_USD'],
    mode='lines+markers', name='CLP/USD diario',
    line=dict(color='#90CAF9', width=1.2),
    marker=dict(size=3.5), opacity=0.85
))
fig.add_trace(go.Scatter(
    x=df_hist['Fecha'], y=df_hist['MA20'],
    mode='lines', name=f'MA {WINDOW_MA} días',
    line=dict(color='#1565C0', width=2.8)
))
# Banda histórica ±1σ
fig.add_trace(go.Scatter(
    x=pd.concat([df_hist['Fecha'], df_hist['Fecha'][::-1]]),
    y=pd.concat([df_hist['MA20']+df_hist['STD20'],
                 (df_hist['MA20']-df_hist['STD20'])[::-1]]),
    fill='toself', fillcolor='rgba(21,101,192,0.10)',
    line=dict(color='rgba(0,0,0,0)'), name='Banda ±1σ'
))
fig.add_trace(go.Scatter(
    x=df_proj['Fecha'], y=df_proj['Proyeccion'],
    mode='lines+markers', name='Proyección MA',
    line=dict(color='#FF6F00', width=2.5, dash='dash'),
    marker=dict(size=5, symbol='diamond')
))
# Banda proyección ±1.5σ
fig.add_trace(go.Scatter(
    x=pd.concat([df_proj['Fecha'], df_proj['Fecha'][::-1]]),
    y=pd.concat([df_proj['Upper'], df_proj['Lower'][::-1]]),
    fill='toself', fillcolor='rgba(255,111,0,0.12)',
    line=dict(color='rgba(0,0,0,0)'), name='Intervalo ±1.5σ'
))

fig.update_layout(
    title=dict(
        text=(f'CLP/USD — Serie Diaria {YEAR_DATOS} + Proyección 3 meses<br>'
              f'<sup>Fuente: Banco Central de Chile (scraping público)</sup>'),
        font=dict(size=16)
    ),
    yaxis=dict(title='CLP por 1 USD', tickformat='$,.2f'),
    xaxis=dict(title='Fecha'),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)'),
    hovermode='x unified',
    template='plotly_white',
    height=500
)
fig.show()
print("✅ Gráfico Plotly interactivo generado")


## 🏁 Resumen final

In [ ]:
# ─── RESUMEN FINAL ────────────────────────────────────────────────────────────
sep = "─" * 60
print("=" * 60)
print("   ANÁLISIS TIPO DE CAMBIO OBSERVADO CLP/USD")
print("   Fuente: Banco Central de Chile (scraping público)")
print("=" * 60)
print(f"  Año analizado : {YEAR_DATOS}")
print(f"  Período       : {df_long['Fecha'].min():%d-%b-%Y}  →  {df_long['Fecha'].max():%d-%b-%Y}")
print(f"  Registros     : {len(df_long):,} días hábiles")
print()
print("  ESTADÍSTICAS HISTÓRICAS:")
print(f"    Promedio    : ${df_long['CLP_USD'].mean():,.2f} CLP/USD")
print(f"    Mínimo      : ${df_long['CLP_USD'].min():,.2f}  ({df_long.loc[df_long['CLP_USD'].idxmin(),'Fecha']:%d-%b-%Y})")
print(f"    Máximo      : ${df_long['CLP_USD'].max():,.2f}  ({df_long.loc[df_long['CLP_USD'].idxmax(),'Fecha']:%d-%b-%Y})")
print(f"    Volatilidad : ±{df_long['CLP_USD'].std():,.2f} CLP")
print()
print("  PROYECCIÓN 3 MESES:")
print(f"    Estimado    : ${proj_vals[-1]:,.2f} CLP/USD")
print(f"    Rango       : ${proj_lower[-1]:,.2f}  —  ${proj_upper[-1]:,.2f}")
print(f"    Tendencia   : {slope:+.3f} CLP/día  ({slope*20:+.1f} CLP/mes)")
print()
print(sep)
print("  FLUJO DEMOSTRADO:")
print("  1. Scraping  → requests + Selenium fallback")
print("  2. Selección → tabla de meses detectada automáticamente")
print("  3. Limpieza  → ÷100 para obtener tasa CLP/USD real")
print("  4. UNPIVOT   → melt() convierte Día×Mes en (Fecha, CLP_USD)")
print("  5. Viz       → serie diaria · heatmap · barras mensuales")
print("  6. Predicción→ MA-20 + tendencia lineal → 3 meses forward")
print(sep)
